# KIT LV/RV development calibration and diagnosis

Run in the user's Edge Colab CPU runtime. The numerical steps were executed on 2026-10-21 with Python 3.13.16, 186 patient/workbench tests passing, and seven pinned repositories. This consolidated recipe uses a unique output directory to preserve previous runs. Its final code cells are compiled separately; the complete consolidated file is not claimed to have been replayed end to end after consolidation.

One volunteer, previously seen data. KIT v1.1: https://zenodo.org/records/10526554, CC BY-NC 4.0. No clinical or independent patient validation. LV outer data remain a development reassessment. The sparse RV assessment point is explicitly consumed for calibration in the joint experiment; matching it is not an RV validation pass. Pressure scale and several fixed model assumptions remain unidentified. Original failed benchmarks are retained.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
PIN = '1de054aa5fee81b5bb7817410d55b5c0fa25cbf7'
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ROOT = Path('/content/HeartTwin_RV_Diagnosis_' + PIN[:12] + '_' + RUN_ID)
ROOT.mkdir(exist_ok=True)
LOGS = ROOT/'setup-logs'; LOGS.mkdir(exist_ok=True)
ENV = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
def run(command, name, timeout=2400, cwd=None):
    print('Starting', name, flush=True)
    log_path = LOGS/(name+'.log')
    with log_path.open('w') as log:
        result = subprocess.run(list(map(str, command)), cwd=cwd, env=ENV,
            stdout=log, stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        print(log_path.read_text()[-10000:])
        raise RuntimeError(name+' failed; retain the log')
PINS = {
 'Virelion-HeartTwin': PIN,
 'Virelion-CardiMech': '5dc14d886ccbda5a1398b863f9ae7821bbffa18b',
 'Virelion-CardiFlow': '670d370635ba09b2908f7e6736299719e86fc36b',
 'Virelion-CardiInfer': 'b4f0c3c30da5986c6a98dba6ceccf1e1de95c881',
 'Virelion-CardiEP': 'f9bfcb78cdb857e5995ce244d8645f7370d6b3bb',
 'Virelion-CardiAnatomy': 'e3cc3450c49b84334dd2da696f33a75f4f61557f',
 'Virelion-ElectroTrace': 'a4cf37ad2e1ed6719a23befa2be5e47af4b4d1de'}
for name, pin in PINS.items():
    path = ROOT/name
    if not path.exists():
        run(['git','init',path], 'init-'+name)
        run(['git','-C',path,'fetch','--depth','1',
             'https://github.com/Virelion-Biotech/'+name+'.git',pin], 'fetch-'+name)
        run(['git','-C',path,'checkout','--detach','FETCH_HEAD'], 'checkout-'+name)
    assert subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()==pin
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
venv = ROOT/'venv'
if not venv.exists():
    run([sys.executable,'-m','venv','--without-pip',venv], 'venv')
python = venv/'bin/python'
ENV['PATH'] = str(python.parent)+os.pathsep+ENV['PATH']
dependency_path=ROOT/'replay-dependencies.txt'
dependency_path.write_text("annotated-types==0.8.0\nattrs==26.1.0\nblinker==1.9.0\nclick==8.5.0\ncloudpickle==3.1.2\ncontourpy==1.4.0\ncycler==0.12.1\nFlask==3.1.3\nfonttools==4.67.0\nhypothesis==6.170.5\niniconfig==2.3.1\nitsdangerous==2.2.0\nJinja2==3.1.6\njoblib==1.6.0\njsonschema==4.25.1\njsonschema-specifications==2025.9.1\nkiwisolver==1.5.1\nmarkdown-it-py==4.2.0\nMarkupSafe==3.0.4\nmatplotlib==3.10.7\nmdurl==0.1.2\nmeshio==5.3.5\nnarwhals==2.27.1\nnibabel==5.3.2\nnumpy==2.2.6\npackaging==26.3\npandas==3.0.6\npillow==13.0.0\npluggy==1.7.0\npydantic==2.12.3\npydantic_core==2.41.4\npydicom==3.0.1\nPygments==2.21.0\npyparsing==3.3.3\npytest==8.4.2\npython-dateutil==2.9.0.post0\nPyYAML==6.0.3\nreferencing==0.37.0\nrich==15.0.0\nrpds-py==2026.9.1\nruff==0.14.1\nscikit-learn==1.9.1\nscipy==1.16.3\nsix==1.17.0\nsortedcontainers==2.4.0\nthreadpoolctl==3.7.0\ntyping-inspection==0.4.4\ntyping_extensions==4.16.0\nWerkzeug==3.1.9\n")
run([sys.executable,'-m','pip','--python',python,'install','pip==25.1.1','-r',dependency_path],'dependencies')
for name in PINS:
    run([python,'-m','pip','install','-e',ROOT/name], 'install-'+name)
run([python,'-m','pip','check'], 'pip-check')
run([python,'-m','pip','freeze'], 'environment')
(LOGS/'pins.json').write_text(json.dumps(PINS,indent=2))
repo = ROOT/'Virelion-HeartTwin'
baseline = repo/'validation/kit_edgar_2026_10_20'
OUT = ROOT/'rv-results'; OUT.mkdir(exist_ok=True)
print('Pinned RV diagnostic environment ready')


## Software checks

The standalone research fit uses SciPy and HeartTwin's native circulation. Registry and command routing are tested separately. It does not claim a passed clinical PatientCase evidence gate or independent source partition; the LV split shares one acquisition. The lifecycle suite includes a real CardiInfer synthetic inverse/replay regression.


In [ ]:
run([python,'-m','py_compile',repo/'scripts/diagnose_kit_rv.py',repo/'scripts/fit_kit_joint_lv_rv.py'],'runner-compile',cwd=repo)
run([python,'-m','ruff','check',repo/'scripts/diagnose_kit_rv.py',repo/'scripts/fit_kit_joint_lv_rv.py'],'runner-lint',cwd=repo)
run([python,repo/'scripts/validate_patient_lifecycle.py','--output',ROOT/'lifecycle'],'patient-lifecycle',cwd=repo)
print((ROOT/'lifecycle/STATUS.json').read_text())


## Original RV diagnosis and conditional controls

The independent solver includes nonlinear and asymmetric chamber laws. All measured timestamps remain unchanged. Optional strength fits consume the previously seen RV point for development fitting; source requests are immutable. Their LV side effects motivate joint calibration.


In [ ]:
run([python,repo/'scripts/diagnose_kit_rv.py','--baseline-dir',baseline,
 '--request',baseline/'fitted_model.json','--output',OUT,'--constraint-fit'],'rv-diagnosis',cwd=repo)
post_path=ROOT/'postprocess_rv.py'
post_path.write_text("import importlib.util, json\nfrom copy import deepcopy\nfrom pathlib import Path\nimport sys\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom hearttwin.patient_closed_loop import COMPARTMENTS, simulate_closed_loop\nroot,repo=map(Path,sys.argv[1:3]); out=root/'rv-results'\nspec=importlib.util.spec_from_file_location('rv_diagnostic',repo/'scripts/diagnose_kit_rv.py')\nmodule=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\nrequest=json.loads((out/'diagnostic_request.json').read_text())\nnative=json.loads((out/'diagnostic_prediction.json').read_text())\ntimes=np.asarray(native['time_s'])\nreference=module.reference(request)\noriginal=np.column_stack([native['volume_mL'][c] for c in COMPARTMENTS])\ntight_request=deepcopy(request)\ntight_request['solver']={'rtol':1e-9,'atol_mL':1e-10,'max_step_s':.0005}\ntight_request['output_dir']=str(out/'artifacts/tolerance-refinement')\ntight=simulate_closed_loop(tight_request)\ntight_values=np.column_stack([tight['volume_mL'][c] for c in COMPARTMENTS])\nerror=np.abs(original-reference); index=np.unravel_index(np.argmax(error),error.shape)\ntarget=json.loads((out/'solver_transport_checks.json').read_text())['RV_target_time_s']\ntarget_index=np.flatnonzero(times==target)[0]\nreport={'original_native_vs_reference_max_mL':float(error.max()),\n'maximum_error_compartment':COMPARTMENTS[index[1]],'maximum_error_time_s':float(times[index[0]]),\n'RV_at_target_native_minus_reference_mL':float(original[target_index,5]-reference[target_index,5]),\n'tight_native_vs_reference_max_mL':float(np.max(np.abs(tight_values-reference))),\n'tight_vs_original_max_mL':float(np.max(np.abs(tight_values-original))),\n'RV_at_target_tight_minus_reference_mL':float(tight_values[target_index,5]-reference[target_index,5]),\n'interpretation':'Dense-output tolerance error is checked separately from 21.87 mL RV model discrepancy; this is numerical consistency, not patient validation'}\n(out/'tolerance_refinement.json').write_text(json.dumps(report,indent=2))\nfig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True,constrained_layout=True)\naxes[0].plot(times,native['volume_mL']['RV'],color='#324761',lw=2.2,label='Original RV assumptions')\ncolors=['#128a83','#cc8144','#8b5d9c']\nfor factor,color in zip([.5,1.,2.],colors):\n p=json.loads((out/('constraint-'+str(factor)+'-prediction.json')).read_text())\n label='Fit to RV point; outflow resistance x'+str(factor)\n axes[0].plot(p['time_s'],p['volume_mL']['RV'],color=color,lw=1.3,label=label)\n axes[1].plot(p['time_s'],np.asarray(p['volume_mL']['LV'])-native['volume_mL']['LV'],color=color,lw=1.5,label='Resistance x'+str(factor))\naxes[0].scatter([0.,target],[194.545622,65.938406],color='#c2383b',s=55,zorder=5,label='Two observed RV volumes')\naxes[0].set_ylabel('RV volume (mL)')\naxes[0].legend(fontsize=8,loc='upper right')\naxes[1].axhline(0,color='#64748b',lw=.7)\naxes[1].set_ylabel('Change in LV volume (mL)')\naxes[1].set_xlabel('Measured time (s)')\naxes[1].legend(fontsize=8)\nfor ax in axes:\n ax.grid(alpha=.15);ax.spines[['top','right']].set_visible(False)\nfig.suptitle('One RV point admits different conditional fits; LV behavior changes',fontsize=12)\nfig.supxlabel('Previously seen data, one volunteer. Conditional fits are not validation or identified patient parameters.',fontsize=9)\nfig.savefig(out/'RV_diagnosis.png',dpi=170)\nfig.savefig(out/'RV_diagnosis.svg')\nprint(json.dumps(report,indent=2))\n")
run([python,post_path,ROOT,repo],'rv-tolerance-and-figures',cwd=repo)
print((OUT/'solver_transport_checks.json').read_text())
print((OUT/'tolerance_refinement.json').read_text())


## Reproduce all four LV families

Two workers reproduce the prior ablations and stiffness profiles. Each fits the first 17 original calibration samples, selects on the next six, then refits all 23. Cross-family choice uses inner LV error, never the previously seen outer score. All families and unsuccessful trials remain in their files. No new phase search or timestamp warp is added.


In [ ]:
CHROOT=ROOT/'chamber-rebuild'; CHROOT.mkdir(exist_ok=True)
driver_path=ROOT/'rebuild_chamber_evidence.py'
driver_path.write_text("from concurrent.futures import ThreadPoolExecutor, as_completed\nimport hashlib, json, os\nfrom pathlib import Path\nimport shutil, subprocess, sys, time\n\nroot=Path(sys.argv[1]); repo=Path(sys.argv[2]); baseline=repo/'validation/kit_edgar_2026_10_20'\nroot.mkdir(exist_ok=True)\nfamilies=[\n ('constitutive-first','results_chamber_laws','diagnose_kit_chamber_laws.py',[]),\n ('bound-refinement','results_chamber_refinement','diagnose_kit_chamber_laws.py',['--refine-calibration']),\n ('profile-phase','results_stiffness_profile','profile_kit_passive_stiffness.py',[]),\n ('profile-fixed-phase','results_stiffness_fixed_phase','profile_kit_passive_stiffness.py',['--fixed-phase'])]\nprotocol={'scope':'Regeneration of prior development evidence; no unseen data',\n'python':sys.version,'source_pin':subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip(),\n'families':families,'workers':2,'selection':'Lowest inner calibration RMSE; never archived outer error',\n'outer_values_previously_seen':True,'independent_patient_validation':False,\n'clinical_validation':'NOT_ESTABLISHED'}\n(root/'rebuild_protocol.json').write_text(json.dumps(protocol,indent=2))\nenv=dict(os.environ,MPLBACKEND='Agg',PYTHONUNBUFFERED='1',OPENBLAS_NUM_THREADS='1')\ndef worker(row):\n name,folder,script,extra=row\n started=time.monotonic()\n command=[sys.executable,str(repo/'scripts'/script),'--baseline',str(baseline),'--output',str(root/folder),*extra]\n with (root/(name+'.log')).open('w') as log:\n  try:\n   completed=subprocess.run(command,cwd=repo,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=3600)\n   code=completed.returncode\n  except subprocess.TimeoutExpired:\n   code=124\n return {'family':name,'folder':folder,'exit_code':code,'seconds':time.monotonic()-started}\nrows=[]\nwith ThreadPoolExecutor(max_workers=2) as pool:\n for result in as_completed([pool.submit(worker,row) for row in families]):\n  rows.append(result.result())\n  (root/'process_status.json').write_text(json.dumps(rows,indent=2))\n  print('FINISHED',json.dumps(rows[-1]),flush=True)\nif any(row['exit_code'] for row in rows):\n raise RuntimeError('A family failed; retain all logs and diagnose before selection')\nranking=[]\nfor name,folder,_,_ in families:\n selected=json.loads((root/folder/'selection.json').read_text())\n ranking.append({'family':name,'folder':folder,'inner_rmse_mL':selected['inner_rmse_mL']})\nranking.sort(key=lambda x:(x['inner_rmse_mL'],x['family']))\nfinal=root/'results_chamber_final'; final.mkdir(exist_ok=True)\nchoice={'rule':'Lowest inner RMSE across all recorded families; outer never used for choice',\n'prior_outer_values_seen':True,'independent_patient_validation':False,'ranking':ranking,\n'selected_family':ranking[0]['family']}\n(final/'cross_round_selection.json').write_text(json.dumps(choice,indent=2))\nselected_dir=root/ranking[0]['folder']\nfor source,target in [('results.json','selected_results.json'),('frozen_request.json','selected_request.json'),\n                      ('frozen_prediction.json','selected_prediction.json')]:\n shutil.copyfile(selected_dir/source,final/target)\nchoice['prediction_sha256']=hashlib.sha256((final/'selected_prediction.json').read_bytes()).hexdigest()\n(final/'cross_round_selection.json').write_text(json.dumps(choice,indent=2))\nprint('SELECTED',json.dumps(choice),flush=True)\nprint('RESULT', (final/'selected_results.json').read_text(),flush=True)\n")
run([python,driver_path,CHROOT,repo],'chamber-rebuild',timeout=7200,cwd=repo)
print((CHROOT/'results_chamber_final/cross_round_selection.json').read_text())
print((CHROOT/'results_chamber_final/selected_results.json').read_text())


## Diagnose the exact selected LV request

Repeat numerical/routing and sparse-RV sensitivity checks against the complete frozen candidate, with no partial reconstruction or silent fallback. Constraint fits remain separate from the selected LV-only model.


In [ ]:
SELECTED_REQUEST=CHROOT/'results_chamber_final/selected_request.json'
run([python,repo/'scripts/diagnose_kit_rv.py','--baseline-dir',baseline,'--request',SELECTED_REQUEST,
 '--output',ROOT/'rv-selected-LV','--constraint-fit'],'rv-selected-LV-diagnosis',cwd=repo)
selected_post=ROOT/'postprocess_rv_selected.py'
selected_post.write_text("import importlib.util, json\nfrom copy import deepcopy\nfrom pathlib import Path\nimport sys\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom hearttwin.patient_closed_loop import COMPARTMENTS, simulate_closed_loop\nroot,repo=map(Path,sys.argv[1:3]); out=root/'rv-selected-LV'\nspec=importlib.util.spec_from_file_location('rv_diagnostic',repo/'scripts/diagnose_kit_rv.py')\nmodule=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\nrequest=json.loads((out/'diagnostic_request.json').read_text())\nnative=json.loads((out/'diagnostic_prediction.json').read_text())\ntimes=np.asarray(native['time_s'])\nreference=module.reference(request)\noriginal=np.column_stack([native['volume_mL'][c] for c in COMPARTMENTS])\ntight_request=deepcopy(request)\ntight_request['solver']={'rtol':1e-9,'atol_mL':1e-10,'max_step_s':.0005}\ntight_request['output_dir']=str(out/'artifacts/tolerance-refinement')\ntight=simulate_closed_loop(tight_request)\ntight_values=np.column_stack([tight['volume_mL'][c] for c in COMPARTMENTS])\nerror=np.abs(original-reference); index=np.unravel_index(np.argmax(error),error.shape)\ntarget=json.loads((out/'solver_transport_checks.json').read_text())['RV_target_time_s']\ntarget_index=np.flatnonzero(times==target)[0]\nreport={'original_native_vs_reference_max_mL':float(error.max()),\n'maximum_error_compartment':COMPARTMENTS[index[1]],'maximum_error_time_s':float(times[index[0]]),\n'RV_at_target_native_minus_reference_mL':float(original[target_index,5]-reference[target_index,5]),\n'tight_native_vs_reference_max_mL':float(np.max(np.abs(tight_values-reference))),\n'tight_vs_original_max_mL':float(np.max(np.abs(tight_values-original))),\n'RV_at_target_tight_minus_reference_mL':float(tight_values[target_index,5]-reference[target_index,5]),\n'interpretation':'Dense-output tolerance error is checked separately from 21.87 mL RV model discrepancy; this is numerical consistency, not patient validation'}\n(out/'tolerance_refinement.json').write_text(json.dumps(report,indent=2))\nfig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True,constrained_layout=True)\naxes[0].plot(times,native['volume_mL']['RV'],color='#324761',lw=2.2,label='LV candidate; original RV assumptions')\ncolors=['#128a83','#cc8144','#8b5d9c']\nfor factor,color in zip([.5,1.,2.],colors):\n p=json.loads((out/('constraint-'+str(factor)+'-prediction.json')).read_text())\n label='Fit to RV point; outflow resistance x'+str(factor)\n axes[0].plot(p['time_s'],p['volume_mL']['RV'],color=color,lw=1.3,label=label)\n axes[1].plot(p['time_s'],np.asarray(p['volume_mL']['LV'])-native['volume_mL']['LV'],color=color,lw=1.5,label='Resistance x'+str(factor))\naxes[0].scatter([0.,target],[194.545622,65.938406],color='#c2383b',s=55,zorder=5,label='Two observed RV volumes')\naxes[0].set_ylabel('RV volume (mL)')\naxes[0].legend(fontsize=8,loc='upper right')\naxes[1].axhline(0,color='#64748b',lw=.7)\naxes[1].set_ylabel('Change in LV volume (mL)')\naxes[1].set_xlabel('Measured time (s)')\naxes[1].legend(fontsize=8)\nfor ax in axes:\n ax.grid(alpha=.15);ax.spines[['top','right']].set_visible(False)\nfig.suptitle('Selected LV model: conditional RV fits still change LV behavior',fontsize=12)\nfig.supxlabel('Previously seen data, one volunteer. Conditional fits are not validation or identified patient parameters.',fontsize=9)\nfig.savefig(out/'RV_diagnosis.png',dpi=170)\nfig.savefig(out/'RV_diagnosis.svg')\nprint(json.dumps(report,indent=2))\n")
run([python,selected_post,ROOT,repo],'rv-selected-tolerance-and-figures',cwd=repo)
comparison_path=ROOT/'plot_lv_rv_comparison.py'
comparison_path.write_text("from copy import deepcopy\nimport json\nfrom pathlib import Path\nimport sys\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom hearttwin.patient_closed_loop import COMPARTMENTS, simulate_closed_loop\nroot,repo=map(Path,sys.argv[1:3]); base=repo/'validation/kit_edgar_2026_10_20'\nout=root/'chamber-rebuild/results_chamber_final'\ncal=json.loads((base/'observations/LV_calibration.json').read_text())\nouter=json.loads((base/'observations/LV_temporal_holdout.json').read_text())\nt=np.array(cal['time_s']+outer['time_s']); y=np.array(cal['values']+outer['values'])\noriginal=json.loads((base/'frozen_prediction.json').read_text())\nselected=json.loads((out/'selected_prediction.json').read_text())\nconditional=json.loads((root/'rv-selected-LV/constraint-1.0-prediction.json').read_text())\nseries={'Original failed LV fit':np.interp(t,original['time_s'],original['volume_mL']['LV']),\n'Selected LV-only development fit':np.interp(t,selected['time_s'],selected['volume_mL']['LV']),\n'RV constraint; no joint LV refit':np.interp(t,conditional['time_s'],conditional['volume_mL']['LV'])}\nmetrics={}\nfor name,prediction in series.items():\n error=prediction-y\n metrics[name]={'calibration_RMSE_mL':float(np.sqrt(np.mean(error[:23]**2))),\n 'outer_development_RMSE_mL':float(np.sqrt(np.mean(error[23:]**2))),\n 'outer_bias_mL':float(error[23:].mean()),'outer_MAE_mL':float(np.mean(np.abs(error[23:]))),\n 'outer_max_absolute_error_mL':float(np.max(np.abs(error[23:])))}\nrequest=json.loads((out/'selected_request.json').read_text())\nscale_checks=[]\nfor scale in [.5,2.]:\n p=deepcopy(request);p['output_dir']=str(out/'pressure-scale-artifacts')\n p['edge_resistance_mmHg_s_per_mL']=(np.array(p['edge_resistance_mmHg_s_per_mL'])*scale).tolist()\n for part in p['compartments'].values():\n  part['external_pressure_mmHg']*=scale\n  if 'emax_mmHg_per_mL' in part:\n   part['emin_mmHg_per_mL']*=scale;part['emax_mmHg_per_mL']*=scale\n  else:part['compliance_mL_per_mmHg']/=scale\n alternative=simulate_closed_loop(p)\n scale_checks.append({'scale':scale,'maximum_volume_change_mL':max(float(np.max(np.abs(\n  np.array(alternative['volume_mL'][c])-selected['volume_mL'][c]))) for c in COMPARTMENTS),\n  'LV_peak_pressure_mmHg':float(max(alternative['pressure_mmHg']['LV']))})\nreport={'metrics':metrics,'pressure_scale_checks':scale_checks,\n'interpretation':'Same previously seen data. RV constraint fits its former assessment point; no LV refit. No independent or clinical validation.'}\n(out/'comparison_metrics.json').write_text(json.dumps(report,indent=2))\nfig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True,constrained_layout=True)\ncolors=['#6c7d8f','#d06b32','#8356a1']\nfor (name,prediction),color in zip(series.items(),colors):\n axes[0].plot(t,prediction,color=color,lw=1.8,label=name)\n axes[1].plot(t,prediction-y,color=color,lw=1.5)\naxes[0].scatter(t,y,s=20,c='#172c3c',zorder=4,label='Observed LV volumes')\nboundary=(t[22]+t[23])/2\nfor ax in axes:\n ax.axvspan(boundary,t[-1],color='#d4dee7',alpha=.35)\n ax.axvline(boundary,color='#7790a5',ls='--',lw=.8)\n ax.grid(alpha=.15);ax.spines[['top','right']].set_visible(False)\naxes[0].set_ylabel('LV volume (mL)');axes[0].legend(fontsize=8)\naxes[0].text(.82,.98,'Previously seen outer data',ha='left',va='top',fontsize=8,\n transform=axes[0].get_xaxis_transform())\naxes[1].axhline(0,color='#172c3c',lw=.8);axes[1].set_ylabel('Predicted minus observed (mL)')\naxes[1].set_xlabel('Measured time (s)')\nfig.suptitle('LV development error reproduced: 22.87 to 6.63 mL; RV adjustment changes the fit',fontsize=12)\nfig.supxlabel('One volunteer; original calibration split retained. Reproduction is not independent validation.',fontsize=9)\nfig.savefig(out/'LV_RV_development_comparison.png',dpi=170)\nfig.savefig(out/'LV_RV_development_comparison.svg')\nprint(json.dumps(report,indent=2))\n")
run([python,comparison_path,ROOT,repo],'lv-rv-comparison',cwd=repo)
print((CHROOT/'results_chamber_final/comparison_metrics.json').read_text())


## Conditional joint LV/RV calibration

Seven coordinates are fitted jointly under each of three fixed RV outflow resistances. Curvature k stays at the previously selected LV value; other loading/reference assumptions remain fixed. The objective is unweighted LV and RV volume SSE in mL, not a validated noise model. The RV initial volume is an initial condition, and the one subsequent RV point is calibration data. Select the joint loading family by six inner LV samples, then refit all 23 LV samples plus the RV point. Freeze every candidate before outer development rescoring. This is a separate joint research candidate; it does not retrospectively turn an assessment point into a held-out pass or identify true loading/pressure.


In [ ]:
run([python,repo/'scripts/fit_kit_joint_lv_rv.py','--baseline',baseline,'--candidate',SELECTED_REQUEST,
 '--output',ROOT/'joint-lv-rv'],'joint-fit',timeout=3600,cwd=repo)
joint_results=json.loads((ROOT/'joint-lv-rv/results.json').read_text())
joint_factor=joint_results['selected_loading_factor']
JOINT_REQUEST=ROOT/'joint-lv-rv'/('loading-'+str(joint_factor))/'frozen_request.json'
run([python,repo/'scripts/diagnose_kit_rv.py','--baseline-dir',baseline,'--request',JOINT_REQUEST,
 '--output',ROOT/'joint-verification'],'joint-verification',cwd=repo)
joint_post=ROOT/'postprocess_joint_fit.py'
joint_post.write_text("import importlib.util, json\nfrom copy import deepcopy\nfrom pathlib import Path\nimport sys\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom hearttwin.patient_closed_loop import COMPARTMENTS, simulate_closed_loop\nroot,repo=map(Path,sys.argv[1:3]); out=root/'joint-verification'\nspec=importlib.util.spec_from_file_location('rv_diagnostic',repo/'scripts/diagnose_kit_rv.py')\nmodule=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\nrequest=json.loads((out/'diagnostic_request.json').read_text())\nnative=json.loads((out/'diagnostic_prediction.json').read_text())\ntimes=np.asarray(native['time_s'])\nreference=module.reference(request)\noriginal=np.column_stack([native['volume_mL'][c] for c in COMPARTMENTS])\ntight_request=deepcopy(request)\ntight_request['solver']={'rtol':1e-9,'atol_mL':1e-10,'max_step_s':.0005}\ntight_request['output_dir']=str(out/'artifacts/tolerance-refinement')\ntight=simulate_closed_loop(tight_request)\ntight_values=np.column_stack([tight['volume_mL'][c] for c in COMPARTMENTS])\nerror=np.abs(original-reference); index=np.unravel_index(np.argmax(error),error.shape)\ntarget=json.loads((out/'solver_transport_checks.json').read_text())['RV_target_time_s']\ntarget_index=np.flatnonzero(times==target)[0]\nreport={'original_native_vs_reference_max_mL':float(error.max()),\n'maximum_error_compartment':COMPARTMENTS[index[1]],'maximum_error_time_s':float(times[index[0]]),\n'RV_at_target_native_minus_reference_mL':float(original[target_index,5]-reference[target_index,5]),\n'tight_native_vs_reference_max_mL':float(np.max(np.abs(tight_values-reference))),\n'tight_vs_original_max_mL':float(np.max(np.abs(tight_values-original))),\n'RV_at_target_tight_minus_reference_mL':float(tight_values[target_index,5]-reference[target_index,5]),\n'interpretation':'Dense-output tolerance error is checked separately from 21.87 mL RV model discrepancy; this is numerical consistency, not patient validation'}\n(out/'tolerance_refinement.json').write_text(json.dumps(report,indent=2))\n\nbase=repo/'validation/kit_edgar_2026_10_20'\njoint_root=root/'joint-lv-rv'\njoint_results=json.loads((joint_root/'results.json').read_text())\nfactor=joint_results['selected_loading_factor']\njoint=json.loads((joint_root/('loading-'+str(factor))/'frozen_prediction.json').read_text())\nlv_only=json.loads((root/'chamber-rebuild/results_chamber_final/selected_prediction.json').read_text())\noriginal=json.loads((base/'frozen_prediction.json').read_text())\ncal=json.loads((base/'observations/LV_calibration.json').read_text())\nouter=json.loads((base/'observations/LV_temporal_holdout.json').read_text())\nobserved_t=np.array(cal['time_s']+outer['time_s'])\nobserved_y=np.array(cal['values']+outer['values'])\nfig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True,constrained_layout=True)\nfor name,prediction,color,lw in [\n ('Original failed LV fit',original,'#8a99a7',1.1),\n ('LV-only selected fit',lv_only,'#d06b32',1.6),\n ('Conditional joint LV/RV fit',joint,'#177e89',2.0)]:\n axes[0].plot(prediction['time_s'],prediction['volume_mL']['LV'],color=color,lw=lw,label=name)\naxes[0].scatter(observed_t,observed_y,color='#172c3c',s=18,zorder=4,label='Observed LV volumes')\nboundary=(observed_t[22]+observed_t[23])/2\naxes[0].axvspan(boundary,observed_t[-1],color='#d4dee7',alpha=.35)\naxes[0].axvline(boundary,color='#7790a5',ls='--',lw=.8)\naxes[0].set_ylabel('LV volume (mL)')\naxes[0].legend(fontsize=8,loc='upper center')\naxes[0].text(.82,.98,'Previously seen LV outer data',ha='left',va='top',fontsize=8,\n transform=axes[0].get_xaxis_transform())\naxes[1].plot(lv_only['time_s'],lv_only['volume_mL']['RV'],color='#d06b32',lw=1.6,label='LV-only model; RV unfitted')\naxes[1].plot(joint['time_s'],joint['volume_mL']['RV'],color='#177e89',lw=2,label='Conditional joint fit')\naxes[1].scatter([0.,target],[194.545622,65.938406],s=48,color='#b82e38',zorder=4,label='RV initial condition and calibration point')\naxes[1].annotate('Used for calibration',xy=(target,65.938406),xytext=(.45,90),\n arrowprops={'arrowstyle':'-','color':'#7a8490'},fontsize=8)\naxes[1].set_ylabel('RV volume (mL)')\naxes[1].set_xlabel('Measured time (s)')\naxes[1].legend(fontsize=8,loc='upper center')\nfor ax in axes:\n ax.grid(alpha=.15)\n ax.spines[['top','right']].set_visible(False)\nfig.suptitle('Joint development fit: LV outer RMSE 6.50 mL; RV calibration residual 0.465 mL',fontsize=12)\nfig.supxlabel('One volunteer, previously seen data. RV point fitted; no independent RV or clinical validation.',fontsize=9)\nfig.savefig(joint_root/'Joint_LV_RV_development.png',dpi=170)\nfig.savefig(joint_root/'Joint_LV_RV_development.svg')\nimport csv\nwith (joint_root/'LV_comparison.csv').open('w',newline='') as stream:\n writer=csv.writer(stream)\n writer.writerow(['time_s','observed_LV_mL','original_LV_mL','LV_only_mL','joint_LV_mL','partition'])\n predictions=[np.interp(observed_t,p['time_s'],p['volume_mL']['LV']) for p in [original,lv_only,joint]]\n for index,time in enumerate(observed_t):\n  writer.writerow([time,observed_y[index],*[p[index] for p in predictions],\n      'calibration' if index<23 else 'previously_seen_outer'])\nprint(json.dumps(report,indent=2))\nprint('JOINT_POSTPROCESS_COMPLETE')\n")
run([python,joint_post,ROOT,repo],'joint-tolerance-and-figures',cwd=repo)
print((ROOT/'joint-lv-rv/results.json').read_text())
print((ROOT/'joint-verification/solver_transport_checks.json').read_text())
print((ROOT/'joint-verification/tolerance_refinement.json').read_text())


## Export all evidence

Run this cell separately if an earlier numerical cell fails, so its logs are retained. The ZIP includes generated native artifacts, frozen requests/predictions, family controls, figures, source helpers, environment and test XML. Original source geometry is not included. Colab browser downloads may not produce an event observable by automation; verify the ZIP exists and retain its printed SHA256. The Files pane also exposes the archive. A ZIP is not proof of success: inspect statuses, optimizer termination and failed trials.


In [ ]:
paths=sorted(p for folder in ['rv-results','rv-selected-LV','lifecycle','setup-logs',
 'chamber-rebuild','joint-lv-rv','joint-verification','execution-failures']
 for p in (ROOT/folder).rglob('*') if p.is_file())
paths+=sorted(ROOT.glob('*.py'))+sorted(ROOT.glob('*.txt'))
manifest={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
manifest_path=ROOT/'RESULT_SHA256.json'
manifest_path.write_text(json.dumps(manifest,indent=2))
archive=ROOT/'KIT_LV_RV_development_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zipped:
 for p in paths+[manifest_path]:zipped.write(p,str(p.relative_to(ROOT)))
print(json.dumps({'archive':str(archive),'bytes':archive.stat().st_size,
 'sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'files':len(paths)},indent=2))
from google.colab import files
files.download(str(archive))
